# Dorado: Two-Stage Reasoning Post-Training

**First time?** Run `bash setup.sh` in a terminal first, then restart kernel.

In [1]:
import os, sys, subprocess
from pathlib import Path

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Deepspeed Hardware Fix (Bypass missing nvcc) ──
os.environ['DS_BUILD_OPS'] = '0'
import shutil
if os.environ.get('CUDA_HOME') == '/usr/local/cuda' and not shutil.which('nvcc'):
    del os.environ['CUDA_HOME']


# Runtime cache
repo_candidates = ["/home/jovyan/llmpt", os.getcwd()]
repo_root = next((p for p in repo_candidates if os.path.isdir(p)), os.getcwd())
cache_dir = os.path.join(repo_root, ".runtime_cache")
Path(cache_dir).mkdir(parents=True, exist_ok=True)
for key, sub in [("DORADO_RUNTIME_CACHE", ""), ("HF_HOME", "huggingface"),
                  ("HF_HUB_CACHE", "huggingface/hub"), ("HF_DATASETS_CACHE", "huggingface/datasets"),
                  ("TRANSFORMERS_CACHE", "huggingface/transformers"), ("PIP_CACHE_DIR", "pip")]:
    os.environ[key] = os.path.join(cache_dir, sub) if sub else cache_dir

# GPU
try:
    rows = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],
        text=True).strip().splitlines()
    selected = [r.split(",")[0].strip() for r in sorted(rows, key=lambda r: int(r.split(",")[1]), reverse=True)[:8]]
except Exception:
    selected = ["0"]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(selected)

# Path
for p in [os.path.abspath("."), os.path.expanduser("~/llmpt")]:
    if os.path.isdir(os.path.join(p, "dorado")):
        if p not in sys.path: sys.path.insert(0, p)
        os.chdir(p)
        break

import torch, transformers
print(f"torch={torch.__version__}  transformers={transformers.__version__}  GPU={os.environ['CUDA_VISIBLE_DEVICES']}")
if torch.cuda.is_available():
    print(f"🎯 {torch.cuda.get_device_name(0)} ({torch.cuda.device_count()} visible)")

from dorado import (get_profile, PROFILES, build_experiment_grid, make_results_paths,
    run_sft_stage, run_candidate_generation, run_labeling_stage,
    run_dpo_training, run_full_evaluation,
    run_single_experiment, run_all_experiments, cleanup_artifacts,
    clear_gpu, set_random_seeds, cleanup_storage)
print(f"✅ dorado loaded")

torch=2.4.0+cu121  transformers=4.49.0  GPU=0
🎯 NVIDIA GeForce RTX 3090 (1 visible)
✅ dorado loaded


/opt/conda/lib/python3.11/site-packages/transformers/utils/hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [ ]:
PROFILE = "fast"  # "fast" | "full"
OVERRIDES = None

EXPERIMENTS = build_experiment_grid(profile=PROFILE, overrides=OVERRIDES)
_, RESULTS_FILE, CHECKPOINT_FILE = make_results_paths()
cfg = EXPERIMENTS[0]
print(f"{PROFILE}: {cfg['base_model']} | SFT={cfg['sft_samples']} | Math={cfg['math_prompt_count']} | β={cfg['dpo_beta']}")

In [3]:
results_df = run_all_experiments(EXPERIMENTS, RESULTS_FILE, CHECKPOINT_FILE)

/opt/conda/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


RuntimeError: Failed to import trl.trainer.dpo_trainer because of the following error (look up to see its traceback):
CUDA_HOME does not exist, unable to compile CUDA op(s)

In [ ]:
if results_df is not None and not results_df.empty:
    display(results_df)
else:
    print("No results yet.")
cleanup_artifacts()